<p align="center">
  <span style="color:Navy; font-size:200%; font-weight:bold; vertical-align:middle;">
        Estadística y Probabilidad    
  </span>
</p>
<p align="center" style="line-height:1.2;">
  <span style="color:RoyalBlue; font-size:160%;">Tema 3: Probabilidad y distribuciones de probabilidad </span><br/>
  <span style="color:DodgerBlue; font-size:140%;"> Probabilidad marginal, condicional y conjunta</span><br/>
  <span style="font-size:100%;color:forestgreen"> Escuela Nacional de Ciencias de la Tierra  |  Semestre 2027-I</span>
</p>

---

# **<font color="Navy">  Laboratorio 3:  Distribuciones de Probabilidad Paramétricas Contínuas y Discretas </font>**

## Record de Terremotos ocurridos en el mundo

Registros de terremotos de todo el mundo recopilados por el Servicio Geológico de los Estados Unidos. Se incluyen detalles importantes sobre el terremoto, como la distancia, el ángulo de cobertura, la magnitud, la profundidad y la significancia, para describir adecuadamente el terremoto. Además, se incluyen datos sobre las coordenadas geológicas exactas y una descripción relativa de la ubicación del terremoto. Los terremotos recopilados son del último mes.

# Bibliotecas


In [ ]:
# Importar las bibliotecas necesarias
#!pip install --upgrade pandas
#!pip install scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from scipy.stats import binom, poisson, nbinom, hypergeom, multinomial, norm, uniform, geom

## Concepto probabilístico de variable aleatoria

Una **variable aleatoria** (v.a.) es una función que asigna un **número** a cada resultado de un experimento aleatorio. Se escribe con mayúscula ($X$); sus valores concretos, con minúscula ($x$).

- Número de sismos perceptibles en un día → $X \in \{0, 1, 2, \dots\}$ → **discreta** (valores que se pueden contar).
- Profundidad a la que aparece un contacto geológico → $X \in [0, 20]$ m → **continua** (cualquier valor en un intervalo).

### Funciones que describen a una variable aleatoria

| Función | Discreta | Continua |
|---|---|---|
| Probabilidad puntual / densidad | **PMF** $P(X = x)$ | **PDF** $f(x)$ (la probabilidad es un **área**) |
| Acumulada | **CDF** $F(x) = P(X \leq x)$ | **CDF** $F(x) = P(X \leq x)$ |

Y dos resúmenes numéricos:

- **Esperanza** (valor esperado): $E[X] = \sum x \cdot P(X=x)$ — el "promedio a largo plazo".
- **Varianza**: $Var(X) = E[(X - E[X])^2]$ — qué tanto se dispersa alrededor de la esperanza.

## Variable aleatoria discreta

### Ejemplo:

Sea $X$ = número de sismos perceptibles registrados en un día por una estación, con la siguiente distribución:

In [ ]:
pmf_sismos = pd.DataFrame({
    "x": [0, 1, 2, 3, 4],
    "p": [0.45, 0.30, 0.15, 0.07, 0.03]
})

# Un requisito: las probabilidades deben sumar 1 (axioma de normalización)
print("Suma de probabilidades:", pmf_sismos["p"].sum())

pmf_sismos["F"] = pmf_sismos["p"].cumsum()     # función de distribución acumulada
pmf_sismos

In [ ]:
esperanza = (pmf_sismos["x"] * pmf_sismos["p"]).sum()
varianza = (((pmf_sismos["x"] - esperanza) ** 2) * pmf_sismos["p"]).sum()
desviacion = np.sqrt(varianza)

print(f"E[X]   = {esperanza:.3f} sismos/día")
print(f"Var(X) = {varianza:.3f}")
print(f"Desv.  = {desviacion:.3f}")

# P(X >= 2) = 1 - P(X <= 1) = 1 - F(1)
p_2_o_mas = 1 - pmf_sismos.loc[pmf_sismos["x"] == 1, "F"].values[0]
print(f"P(X >= 2) = {p_2_o_mas:.2f}")

La esperanza **no** tiene que ser un valor que la variable pueda tomar 
(aquí es 0.93, pero nunca observaremos "0.93 sismos"): es el promedio al que tiende la variable si repetimos el experimento muchas veces.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))

ejes[0].bar(pmf_sismos["x"], pmf_sismos["p"], color="steelblue")
ejes[0].set_title("PMF: P(X = x)")
ejes[0].set_xlabel("Sismos perceptibles por día (x)")
ejes[0].set_ylabel("Probabilidad")

ejes[1].step(pmf_sismos["x"], pmf_sismos["F"], where="post", color="crimson")
ejes[1].scatter(pmf_sismos["x"], pmf_sismos["F"], color="crimson")
ejes[1].set_title("CDF: F(x) = P(X ≤ x)")
ejes[1].set_xlabel("Sismos perceptibles por día (x)")
ejes[1].set_ylabel("Probabilidad acumulada")

plt.tight_layout()
plt.show()

### Ejercicio para resolver 1

Sea $X$ = número de réplicas de magnitud ≥ 4 que siguen a un sismo fuerte, con esta distribución:

| x | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| P(X = x) | 0.50 | 0.30 | 0.15 | 0.05 |

1. Construye el DataFrame y verifica que las probabilidades suman 1.
2. Calcula $E[X]$, $Var(X)$ y la desviación estándar.
3. Calcula $P(X \geq 2)$ y $P(X \leq 1)$ usando la CDF.
4. Grafica la PMF y la CDF.

In [ ]:
# Tu código aquí

## Variable aleatoria continua

Para una v.a. continua, la probabilidad de que tome **exactamente** un valor es 0; lo que tiene sentido es la probabilidad de caer en un **intervalo**, que se calcula como el **área bajo la PDF**:

$$P(a < X < b) = \int_a^b f(x)\,dx = F(b) - F(a)$$

### Ejemplo:

Sea $X$ la profundidad (m) de un contacto geológico, que puede aparecer en cualquier punto de $[0, 20]$ con la misma probabilidad (distribución **uniforme continua**).

In [ ]:
X = uniform(loc=0, scale=20)    # Uniforme en [0, 20]

p_5_8 = X.cdf(8) - X.cdf(5)
print(f"P(5 < X < 8) = {p_5_8:.3f}   (esperado: 3/20 = {3/20:.3f})")

# P(X = 7 exactamente) = 0; en un intervalo diminuto alrededor de 7 es casi 0:
print(f"P(6.999 < X < 7.001) = {X.cdf(7.001) - X.cdf(6.999):.5f}")

print(f"E[X] = {X.mean():.1f} m   |   Var(X) = {X.var():.2f}")

In [ ]:
x_vals = np.linspace(-2, 22, 400)

plt.figure(figsize=(8, 4))
plt.plot(x_vals, X.pdf(x_vals), color="black")
x_relleno = np.linspace(5, 8, 100)
plt.fill_between(x_relleno, X.pdf(x_relleno), color="orange", alpha=0.6, label="P(5 < X < 8) = área")
plt.xlabel("Profundidad (m)")
plt.ylabel("Densidad f(x)")
plt.title("Uniforme continua en [0, 20]: la probabilidad es un área")
plt.legend()
plt.show()

### Ejercicio para resolver 2

Sea $X$ la distancia (km) entre un epicentro y una estación sismológica, distribuida de forma uniforme en $[0, 50]$.

1. Crea la distribución con `uniform(loc=0, scale=50)`.
2. Calcula $P(X < 12)$ y $P(10 < X < 30)$.
3. Calcula $E[X]$ y $Var(X)$ con `.mean()` y `.var()`. Verifica $E[X]$ con la fórmula $(a+b)/2$.
4. Simula 50,000 valores con `.rvs()` y comprueba que la fracción simulada en $(10, 30)$ se parece al valor teórico.

---
# Distribuciones de probabilidad

Vamos a aplicar los conceptos de distribuciones con `scipy.stats`:

| Método | Qué hace |
|---|---|
| `.pmf(k)` / `.pdf(x)` | Probabilidad puntual (discretas) / densidad (continuas) |
| `.cdf(x)` | $P(X \leq x)$ |
| `.sf(x)` | $P(X > x)$ (función de supervivencia, es `1 - cdf`) |
| `.ppf(q)` | Percentil: el valor $x$ tal que $P(X \leq x) = q$ |
| `.rvs(size=n)` | Genera $n$ valores aleatorios de la distribución |
| `.mean()`, `.var()` | Esperanza y varianza |

### Resumen de las distribuciones de este notebook

| Distribución | ¿Qué cuenta/mide? | Parámetros | En `scipy.stats` |
|---|---|---|---|
| **Binomial** | Éxitos en $n$ ensayos independientes | $n$, $p$ | `binom(n, p)` |
| **Poisson** | Eventos por intervalo de tiempo o espacio | $\lambda$ (tasa media) | `poisson(mu=lam)` |
| **Binomial Negativa** | Fracasos antes del $r$-ésimo éxito | $r$, $p$ | `nbinom(r, p)` |
| **Normal** | Variable continua simétrica "en campana" | $\mu$, $\sigma$ | `norm(loc=mu, scale=sigma)` |

## Distribución Binomial

Se aplica cuando: 
(1) hay $n$ ensayos **independientes**, 

(2) cada uno tiene solo dos resultados (éxito/fracaso) y 

(3) la probabilidad de éxito $p$ es **constante**.

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k} \qquad E[X] = np \qquad Var(X) = np(1-p)$$

### Ejempplo — Pozos productivos

En una zona, cada pozo perforado tiene probabilidad $p = 0.30$ de ser productivo (encontrar agua). Se perforan $n = 10$ pozos independientes. Sea $X$ = número de pozos productivos.

In [ ]:
n, p = 10, 0.30
pozos = binom(n, p)

print(f"P(X = 3)  = {pozos.pmf(3):.4f}")
print(f"P(X >= 5) = {pozos.sf(4):.4f}     (sf(4) = P(X > 4))")
print(f"P(2 <= X <= 4) = {pozos.cdf(4) - pozos.cdf(1):.4f}")
print(f"E[X] = {pozos.mean():.1f}   |   Var(X) = {pozos.var():.2f}")

k = np.arange(0, n + 1)
plt.figure(figsize=(8, 4))
plt.bar(k, pozos.pmf(k), color="steelblue")
plt.xlabel("Número de pozos productivos (k)")
plt.ylabel("P(X = k)")
plt.title("Binomial(n=10, p=0.30)")
plt.show()

### Ejemplo — Estimando $p$ con datos reales

En la práctica, $p$ casi nunca se conoce: se **estima** con datos (probabilidad frecuentista). Con el catálogo real de **1000 sismos de Fiji**, estimemos la probabilidad de que un sismo tenga magnitud ≥ 5.0, y úsala para modelar cuántos de los próximos 20 sismos superarán ese umbral.

*Fuente: dataset `quakes`, paquete `datasets` de R (servido por Rdatasets).*

In [ ]:
url_quakes = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/datasets/quakes.csv"
quakes = pd.read_csv(url_quakes)[["lat", "long", "depth", "mag", "stations"]]
quakes = quakes.rename(columns={"depth": "profundidad_km", "mag": "magnitud", "stations": "num_estaciones"})

p_hat = (quakes["magnitud"] >= 5.0).mean()
print(f"Proporción observada de sismos con magnitud >= 5.0: p̂ = {p_hat:.3f}")

sismos20 = binom(20, p_hat)
print(f"E[X] = {sismos20.mean():.2f} sismos fuertes de cada 20")
print(f"P(X >= 6) = {sismos20.sf(5):.4f}")
print(f"P(X = 0)  = {sismos20.pmf(0):.4f}")

### Ejercicio para resolver 3:

Se toman $n = 15$ muestras de suelo independientes; cada una tiene probabilidad $p = 0.20$ de superar el límite permitido de un contaminante. Sea $X$ el número de muestras que lo superan.

1. Calcula $P(X \leq 2)$ y $P(X \geq 4)$.
2. Calcula $E[X]$ y $Var(X)$.
3. ¿Cuál es el valor **más probable** de $X$? (*Pista: `np.argmax` sobre las PMF de $k = 0, \dots, 15$.*)
4. Grafica la PMF.

In [ ]:
# Tu código aquí

## Distribución de Poisson

Modela el **número de eventos** que ocurren en un intervalo de tiempo o región del espacio, cuando los eventos ocurren de forma independiente y a una tasa media constante $\lambda$:

$$P(X = k) = \frac{e^{-\lambda}\lambda^k}{k!} \qquad E[X] = Var(X) = \lambda$$

Una característica clave: **la media es igual a la varianza**. Esto nos servirá más adelante para decidir si Poisson es una buena opción para unos datos.

> 💡 Si el intervalo cambia de tamaño, $\lambda$ se escala proporcionalmente (el doble de tiempo → el doble de $\lambda$).

### Ejemplo — Sismos por año

En una región ocurren, en promedio, $\lambda = 3.2$ sismos de magnitud ≥ 5 por año.

In [ ]:
lam = 3.2
sismos_anio = poisson(mu=lam)

print(f"P(X = 0)  [año sin sismos fuertes] = {sismos_anio.pmf(0):.4f}")
print(f"P(X >= 5) = {sismos_anio.sf(4):.4f}")
print(f"P(2 <= X <= 4) = {sismos_anio.cdf(4) - sismos_anio.cdf(1):.4f}")

# Para 6 meses la tasa se reduce a la mitad: lambda = 1.6
semestre = poisson(mu=lam / 2)
print(f"\nP(ningún sismo fuerte en 6 meses) = {semestre.pmf(0):.4f}")

k = np.arange(0, 13)
plt.figure(figsize=(8, 4))
plt.bar(k, sismos_anio.pmf(k), color="darkorange")
plt.xlabel("Sismos de magnitud ≥ 5 en un año (k)")
plt.ylabel("P(X = k)")
plt.title("Poisson(λ = 3.2)")
plt.show()

### Ejercicio para resolver :

Un volcán tiene en promedio $\lambda = 0.8$ erupciones por década.

1. Calcula la probabilidad de que en una década **no** haya erupciones, y de que haya **2 o más**.
2. Para un periodo de **25 años** (¿cuál es la nueva $\lambda$?), calcula la probabilidad de que haya **al menos una** erupción.
3. Grafica la PMF para el periodo de 25 años.

## Distribución Binomial Negativa

Mientras que la binomial fija el número de ensayos y cuenta éxitos, la **binomial negativa** hace lo contrario: fija el número de éxitos $r$ que queremos y cuenta cuántos **fracasos** ocurren antes de conseguirlos.

$$P(X = k) = \binom{k + r - 1}{k} p^r (1-p)^k \qquad E[X] = \frac{r(1-p)}{p} \qquad Var(X) = \frac{r(1-p)}{p^2}$$

> ⚠️ En `scipy`, `nbinom(n=r, p=p)` cuenta los **fracasos** antes del $r$-ésimo éxito. El número total de ensayos es $X + r$.

Además, su varianza es **mayor** que su media. Por eso es muy útil para conteos más dispersos que Poisson.

### Ejemplo — ¿Cuántos pozos secos antes de 3 productivos?

Una empresa necesita encontrar $r = 3$ pozos productivos. Cada perforación tiene $p = 0.30$ de éxito. Sea $X$ = número de pozos **secos** antes del tercer productivo.

In [ ]:
r, p = 3, 0.30
secos = nbinom(r, p)

print(f"P(X = 5)  = {secos.pmf(5):.4f}   (exactamente 5 pozos secos)")
print(f"P(X <= 4) = {secos.cdf(4):.4f}")
print(f"E[X] = {secos.mean():.1f} pozos secos   -> total esperado de perforaciones = {secos.mean() + r:.1f}")
print(f"Var(X) = {secos.var():.1f}")

In [ ]:
# Verificación "a mano": simulamos la campaña de perforación paso a paso
np.random.seed(10)
fracasos_sim = []
for _ in range(20_000):
    exitos, fracasos = 0, 0
    while exitos < r:
        if np.random.rand() < p:
            exitos += 1
        else:
            fracasos += 1
    fracasos_sim.append(fracasos)

print(f"Promedio de pozos secos simulado: {np.mean(fracasos_sim):.2f}   (teórico: {secos.mean():.2f})")

k = np.arange(0, 26)
plt.figure(figsize=(9, 4))
plt.hist(fracasos_sim, bins=np.arange(-0.5, 26.5, 1), density=True, alpha=0.5, color="gray", label="Simulación")
plt.plot(k, secos.pmf(k), "o-", color="crimson", label="Binomial Negativa (teórica)")
plt.xlabel("Pozos secos antes del 3er productivo (k)")
plt.ylabel("Probabilidad")
plt.title("Binomial Negativa(r=3, p=0.30)")
plt.legend()
plt.show()

### Ejercicio para resolver:

Un paleontólogo quiere encontrar $r = 2$ sitios con fósiles. Cada sitio explorado tiene probabilidad $p = 0.25$ de tener fósiles.

1. Calcula la probabilidad de explorar exactamente **4 sitios sin fósiles** antes de encontrar el segundo con fósiles.
2. Calcula la probabilidad de que el **total de sitios explorados** (con y sin fósiles) sea **a lo sumo 6**. *(Pista: total = fracasos + r, así que total ≤ 6 equivale a fracasos ≤ 4).*
3. Calcula el número esperado de sitios sin fósiles y el total esperado de sitios.

## Distribución Normal

La distribución **normal** (o gaussiana) describe variables continuas simétricas "en campana", determinadas por su media $\mu$ y su desviación estándar $\sigma$:

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}}\, e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

Resultados útiles:
- **Regla 68-95-99.7:** ~68% de los valores caen a $\pm 1\sigma$ de la media, ~95% a $\pm 2\sigma$, ~99.7% a $\pm 3\sigma$.
- **Estandarización:** $Z = (X - \mu)/\sigma$ mide a cuántas desviaciones estándar está un valor de la media.
- **Percentiles:** `.ppf(q)` da el valor bajo el cual queda una fracción $q$ de la distribución.

### Ejemplo — Plomo en el suelo

Imaginemos un sitio contaminado donde la concentración de plomo sigue aproximadamente una normal con $\mu = 38$ mg/kg y $\sigma = 9$ mg/kg.

In [ ]:
mu, sigma = 38, 9
plomo = norm(loc=mu, scale=sigma)

print(f"P(X > 50)      = {plomo.sf(50):.4f}")
print(f"P(30 < X < 45) = {plomo.cdf(45) - plomo.cdf(30):.4f}")
print(f"Percentil 95   = {plomo.ppf(0.95):.2f} mg/kg   (solo 5% de los puntos supera este valor)")

z = (50 - mu) / sigma
print(f"\nz de 50 mg/kg = {z:.2f}  (50 mg/kg está a {z:.2f} desviaciones estándar por encima de la media)")

print("\nRegla 68-95-99.7:")
for k_sig in [1, 2, 3]:
    print(f"  P(μ-{k_sig}σ < X < μ+{k_sig}σ) = {plomo.cdf(mu + k_sig*sigma) - plomo.cdf(mu - k_sig*sigma):.4f}")

In [ ]:
x_vals = np.linspace(mu - 4*sigma, mu + 4*sigma, 400)
x_cola = np.linspace(50, mu + 4*sigma, 100)

plt.figure(figsize=(8, 4))
plt.plot(x_vals, plomo.pdf(x_vals), color="black")
plt.fill_between(x_cola, plomo.pdf(x_cola), color="crimson", alpha=0.5, label=f"P(X > 50) = {plomo.sf(50):.3f}")
plt.axvline(mu, color="gray", linestyle="--")
plt.xlabel("Concentración de Pb (mg/kg)")
plt.ylabel("Densidad")
plt.title("Normal(μ=38, σ=9)")
plt.legend()
plt.show()

### Ejercicio para resolver: 

La porosidad efectiva de un acuífero sigue una normal con $\mu = 22\%$ y $\sigma = 4\%$.

1. Calcula $P(X < 15)$ y $P(18 < X < 26)$.
2. ¿Qué valor de porosidad es superado solo por el 5% de los puntos del acuífero? (percentil 95).
3. Calcula el valor $z$ de una porosidad de 30% e interprétalo con una frase.
4. Grafica la PDF sombreando el área correspondiente a $P(X < 15)$.

# Aguste de distribuciones de probabilidad a datos reales.

En ocasiones, trabajaremos con bases de datos de terceros. Una forma de cargar estos datos es la siguiente:

In [ ]:
# Proporcionamos el url del archivo que queremos descargar
url = "https://corgis-edu.github.io/corgis/datasets/csv/earthquakes/earthquakes.csv"
# Cargamos los datos como lo hemos hecho anteriormente y lo almacenamos en un Data Frame
data = pd.read_csv(url)

# Mostrar las primeras filas del dataset
data.head(3)

In [ ]:
#Queremos ver cuales son los nombres de las columnas del Data Frame
data.columns

# Descripción de las variabes:

1. **id**: Identificador único del terremoto.
2. **impact.gap**: Brecha azimutal entre estaciones, en grados (0 a 180).
3. **impact.magnitude**: Magnitud del terremoto en la escala de Richter (-1 a 10).
4. **impact.significance**: Significancia del evento (0 a 1000).
5. **location.depth**: Profundidad del terremoto en kilómetros.
6. **location.distance**: Distancia aproximada al epicentro desde la estación de reporte, en grados.
7. **location.full**: Nombre completo de la ubicación.
8. **location.latitude**: Latitud en grados decimales (-90 a 90).
9. **location.longitude**: Longitud en grados decimales (-180 a 180).
10. **location.name**: Nombre del estado o país donde se reportó el terremoto.
11. **time.day**: Día del mes en que ocurrió el terremoto.
12. **time.epoch**: Tiempo en segundos desde el 1 de enero de 1970.
13. **time.full**: Fecha y hora completa del terremoto.
14. **time.hour**: Hora en que ocurrió el terremoto.
15. **time.minute**: Minuto en que ocurrió el terremoto.
16. **time.month**: Mes en que ocurrió el terremoto.
17. **time.second**: Segundo en que ocurrió el terremoto.
18. **time.year**: Año en que ocurrió el terremoto.


## Aplicación Distribución Paramétrica Discreta

In [ ]:
#Trabajaremos con una variable discreta: impact significance
# Esta variable es un número que describe cuán significativo es el evento.
#Los números más grandes indican un evento más significativo.
#Este valor se determina en función de varios factores, incluida la magnitud, el MMI máximo, los informes de personas que lo sintieron y el impacto estimado.
#Varía de 0 a 1000.

# Seleccionar la variable de interés, por ejemplo, la significancia del terremoto
significance = data['impact.significance'].dropna()

In [ ]:
# Veamos una descripción de los datos para saver el valor máximo y el mínimo
significance.describe()

In [ ]:
# Crear el histograma con el rango del eje x de 0 a 1000
plt.figure(figsize=(10, 6))
plt.hist(significance, range=(0, 1000), color='skyblue', edgecolor='black')
plt.xlabel('Significancia')
plt.ylabel('Frecuencia')
plt.title('Ocurrencia de la Significancia de los Terremotos')
plt.grid(True)
plt.show()

In [ ]:
# Ajustaremos la distribución de Poisson a los datos
# Estimamos el parámetro lambda como la media de los datos
lambda_est = significance.mean()
lambda_est

In [ ]:
# Crear el histograma de frecuencias relativas
plt.figure(figsize=(10, 6))
counts, bins, patches = plt.hist(significance, bins=30, range=(0, 1000), density=True, color='skyblue', edgecolor='black', alpha=0.7, label='Frecuencia Relativa')

# Calcular la PDF de la distribución Poisson
x = np.arange(0, 1001)
pdf_poisson = poisson.pmf(x, lambda_est)

# Graficar la PDF de la distribución Poisson como barras
plt.bar(x, pdf_poisson, width=1, color='red', alpha=0.5, label='PDF (Poisson)')

# Etiquetas y título
plt.xlabel('Significancia')
plt.ylabel('Frecuencia Relativa / PDF')
plt.title('Frecuencia Relativa y PDF de la Distribución Poisson para la Significancia de los Terremotos')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Generar valores para la CDF de la distribución Poisson
x = np.arange(0, significance.max() + 1)
cdf_poisson = poisson.cdf(x, lambda_est)

# Frecuencias acumuladas relativas (CDF empírica)
significance_sorted = np.sort(significance)
FrAcRel = np.arange(1, len(significance_sorted) + 1) / len(significance_sorted)

# Graficar las Frecuencias Acumuladas y la CDF
plt.figure(figsize=(10, 6))
plt.plot(significance_sorted, FrAcRel, '.r', linewidth=2, label='FAR (Empírica)')
plt.plot(x, cdf_poisson, 'k', linewidth=2, label='CDF (Poisson)')
plt.xlabel('Significancia')
plt.ylabel('F(x) o Fn(x)')
plt.title('CDF de la Distribución Poisson comparada con la FAR para los datos de significancia', fontsize=10)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Crear el histograma de frecuencias relativas
plt.plot(figsize=(10, 6))
counts, bins, patches = plt.hist(significance, bins=30, range=(0, 1000), density=True, color='skyblue', edgecolor='black', alpha=0.7, label='Frecuencia Relativa')

# Calcular la PDF de la distribución Poisson
x = np.arange(0, 1001)
# poisson.pmf(valores a ajustar, parametro(promedio))
pdf_poisson = poisson.pmf(x, lambda_est)
pdf_geom = geom.pmf(x, prob_templor)

# Graficar la PDF de la distribución Poisson como barras
plt.bar(x, pdf_poisson, width=1, color='red', alpha=0.5, label='PDF (Poisson)')
plt.bar(x, pdf_geom, width=1, color='yellow', alpha=0.5, label='PDF (geom)')
# Etiquetas y título
plt.xlabel('Significancia')
plt.ylabel('Frecuencia Relativa / PDF')
plt.title('Frecuencia Relativa y PDF de la Distribución Poisson para la Significancia de los Terremotos')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Generar valores para la CDF de la distribución Poisson
x = np.arange(0, significance.max() + 1)
cdf_poisson = poisson.cdf(x, lambda_est)

# Frecuencias acumuladas relativas (CDF empírica)
significance_sorted = np.sort(significance)
FrAcRel = np.arange(1, len(significance_sorted) + 1) / len(significance_sorted)

# Graficar las Frecuencias Acumuladas y la CDF
plt.figure(figsize=(10, 6))
plt.plot(significance_sorted, FrAcRel, '.r', linewidth=2, label='FAR (Empírica)')
plt.plot(x, cdf_poisson, 'k', linewidth=2, label='CDF (Poisson)')
plt.xlabel('Significancia')
plt.ylabel('F(x) o Fn(x)')
plt.title('CDF de la Distribución Poisson comparada con la FAR para los datos de significancia', fontsize=10)
plt.legend()
plt.grid(True)
plt.show()

## Ejercicio:
En el siguiente enlace encontrarán una lista de las distribuciones discretas disponibles de la librería scipy.stats
https://docs.scipy.org/doc/scipy/reference/stats.html
(Buscar la sección Discrete distributions)

1. De la lista elegir una distribución.
2. Investigar qué parámetros necesita y cómo se ha usado anteriormente
3. Replicar el procedimiento anterior usando la distribución que eligieron
4. ¿Cuál es mejor?

Nota: necesitarán importar la librería como sigue
from scipy.stats import NombreDeLaDistribucion


# Ahora veámos las una aplicación de las distribuciones contínuas

## En este caso trabajaremos con la variable: Magnitud

La magnitud del terremoto es una medida del tamaño de un terremoto en su fuente. A la misma distancia del terremoto, la amplitud de las ondas sísmicas de las que se determina la magnitud es aproximadamente 10 veces mayor durante un terremoto de magnitud 5 que durante un terremoto de magnitud 4. La cantidad total de energía liberada por el terremoto generalmente aumenta por un factor mayor; para muchos tipos de magnitud comúnmente utilizados, la energía total de un terremoto promedio aumenta por un factor de aproximadamente 32 por cada unidad de aumento en la magnitud. Generalmente varía de -1 (muy pequeño) a 10 (increíblemente poderoso).

In [ ]:
# Seleccionar una columna de interés, por ejemplo, la magnitud de los terremotos
magnitudes = data['impact.magnitude'].dropna()
magnitudes.head()

Primero Hacemos el análisis exploratorio de esta variable.

In [ ]:
#Obtenemos los estadísticos
magnitudes.describe()

In [ ]:
#Obtenemos el histograma de las magnitudes considerando la frencuencia absoluta
sns.histplot(magnitudes, kde=False, stat='count', bins=30)
plt.xlabel('Magnitud')
plt.ylabel('Frecuencias Absoluta')
plt.title('Histograma de Magnitud de Terremotos')
plt.show()

In [ ]:
#Obtenemos el histograma de las magnitudes considerando la frecuncia relativa
# La frecuencia absoluta es el número de veces que se repite algo
# y la frecuencia relativa es la proporción que representa la frecuencia absoluta en relación con el total.

sns.histplot(magnitudes, kde=False, stat='density', bins=30)
#Agregamos la media y la mediana y la moda
plt.axvline(magnitudes.mean(), color='r', linestyle='dashed', linewidth=2, label='Media')
plt.axvline(magnitudes.median(), color='g', linestyle='dashed', linewidth=2, label='Mediana')
plt.axvline(magnitudes.mode()[0], color='b', linestyle='dashed', linewidth=2, label='Moda')
plt.legend()
plt.xlabel('Magnitud')
plt.ylabel('Densidad / Frecuencia relativa')
plt.title('Histograma de Magnitud de Terremotos')
plt.show()

In [ ]:
# Ahora Ajustaremos diferentes distribuciones de probabilidad
#Comensaremos por la Gaussiana o Normal

#Primero obtenemos los parámetros
# Ajustar una distribución normal
params = stats.norm.fit(magnitudes)
#El primer parámetro es la media y el segundo es la mediana
print(params)

# Obtenemos la Función de densidad de probabilidad (pdf)
# norm_pdf = stats.norm.pdf(magnitudes, *params)

# Graficaremos la distribución ajustada en el histograma
sns.histplot(magnitudes, kde=False, stat='density', bins=30)
x = np.linspace(min(magnitudes), max(magnitudes), 100)
plt.plot(x, stats.norm.pdf(x, *params), color='r', label='Distribución Normal')
plt.legend()
plt.show()

In [ ]:
# Número de datos
N = len(magnitudes)

# Frecuencias acumuladas relativas (CDF empírica)
FrAcRel = np.arange(1, N + 1) / N

varx = np.linspace(min(magnitudes), max(magnitudes), 100)
cdf_norm = norm.cdf(varx, *params)

# Ordenar los datos
magnitudes_sort = np.sort(magnitudes)

# Graficar las Frecuencias Acumuladas y la CDF
plt.figure()
plt.plot(magnitudes_sort, FrAcRel, '.r', linewidth=0.5, label='FAR (Empírica)')
plt.plot(varx, cdf_norm, 'k', linewidth=2, label='CDF (Normal)')
plt.xlabel('Magnitud')
plt.ylabel('F(x) o Fn(x)')
plt.title('CDF de la Distribución Normal comparada con la FAR para los datos de magnitud', fontsize=10)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#Ahora vamos a ajustar una distribución exponencial a nuestros datos
#Primero obtenemos los parámetros
# Ajustar una distribución exponencial
params_exp = stats.expon.fit(magnitudes)
print(params_exp)

In [ ]:
# Graficaremos la distribución ajustada en el histograma
sns.histplot(magnitudes, kde=False, stat='density', bins=30)
x = np.linspace(min(magnitudes), max(magnitudes), 100)
plt.plot(x, stats.norm.pdf(x, *params), color='r', label='Distribución Normal')
plt.plot(x, stats.expon.pdf(x, *params_exp), color='g', label='Distribución exponencial')
plt.legend()
plt.show()

## Es su turno de ajustar una distribución:

Su misión es:
1. replicar el procedimiento anterior para ajustar una distribución gamma y una beta
2. Obtener la CDF de la distribución gamma y de la distribución beta
3. ¿Cuál de todas las distribuciones se ajusta mejor?

HINT: stats.gamma.________

In [ ]:
# Les toca ajustar una distribución gamma

# Paso 1: Obtengan los parámetros
params_gamma =

print(params_gamma)

In [ ]:
# Obtén la PDF y graficala en el histograma con las otras distribuciones
sns.histplot(magnitudes, kde=False, stat='density', bins=30)
x = np.linspace(min(magnitudes), max(magnitudes), 100)
plt.plot(x, stats.norm.pdf(x, *params), color='r', label='Distribución Normal')
plt.plot(x, stats.expon.pdf(x, *params_exp), color='g', label='Distribución exponencial')
# Agregue la distribución gamma
plt.plot()
plt.legend()
plt.show()

In [ ]:
#Ahora Obten la CDF y compála con la CDF Normal y los Datos empíricos

# Número de datos
N = len(magnitudes)

# Frecuencias acumuladas relativas (CDF empírica)
FrAcRel = np.arange(1, N + 1) / N

varx = np.linspace(min(magnitudes), max(magnitudes), 100)
#Obtengan la CDF de la distribución
cdf_gamma =
cdf_norm = norm.cdf(varx, *params)

# Ordenar los datos
magnitudes_sort = np.sort(magnitudes)|

# Graficar las Frecuencias Acumuladas y la CDF
plt.figure()
plt.plot(magnitudes_sort, FrAcRel, '.r', linewidth=0.5, label='FAR (Empírica)')
plt.plot(varx, cdf_norm, 'k', linewidth=2, label='CDF (Normal)')
#Agregar la CDF de la distribución Gamma
plt.plot()
plt.xlabel('Magnitud')
plt.ylabel('F(x) o Fn(x)')
plt.title('CDF de la Distribución Normal y Gamma comparada con la FAR para los datos de magnitud', fontsize=10)
plt.legend()
plt.grid(True)
plt.show()